In [1]:
# =========================================
# STEP 1: Install & Import Libraries
# =========================================
!pip install pandas scikit-learn

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# =========================================
# STEP 2: Create User-Item Ratings Dataset
# =========================================
# Rows = Users, Columns = Movies, Values = Ratings
data = {
    'The Matrix':    [5, 4, 0, 0, 3],
    'John Wick':     [4, 5, 0, 0, 2],
    'Titanic':       [0, 0, 5, 4, 0],
    'The Notebook':  [0, 0, 4, 5, 0],
    'Avengers':      [5, 4, 0, 0, 4]
}

df = pd.DataFrame(data, index=[
    'User1', 'User2', 'User3', 'User4', 'User5'
])

print("User-Item Matrix:\n")
print(df)

# =========================================
# STEP 3: Compute User Similarity Matrix
# =========================================
similarity = cosine_similarity(df)

similarity_df = pd.DataFrame(similarity, index=df.index, columns=df.index)

print("\nUser Similarity Matrix:\n")
print(similarity_df)

# =========================================
# STEP 4: Recommendation Function
# =========================================
def recommend_movies(user, df, similarity_df, top_n=3):
    # Get similarity scores for the user
    sim_scores = similarity_df[user]

    # Sort similar users
    sim_scores = sim_scores.sort_values(ascending=False)

    # Remove the user itself
    sim_scores = sim_scores.drop(user)

    # Get top similar users
    top_users = sim_scores.head(2).index

    # Movies already watched by user
    user_rated = df.loc[user]
    unseen_movies = user_rated[user_rated == 0].index

    # Predict scores
    scores = {}

    for movie in unseen_movies:
        weighted_sum = 0
        sim_sum = 0

        for other_user in top_users:
            rating = df.loc[other_user, movie]
            sim = similarity_df.loc[user, other_user]

            if rating > 0:
                weighted_sum += rating * sim
                sim_sum += sim

        if sim_sum != 0:
            scores[movie] = weighted_sum / sim_sum

    # Sort recommendations
    recommended = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [movie for movie, score in recommended[:top_n]]

# =========================================
# STEP 5: Test Recommendation
# =========================================
print("\nRecommendations for User1:")
print(recommend_movies('User1', df, similarity_df))

print("\nRecommendations for User3:")
print(recommend_movies('User3', df, similarity_df))

User-Item Matrix:

       The Matrix  John Wick  Titanic  The Notebook  Avengers
User1           5          4        0             0         5
User2           4          5        0             0         4
User3           0          0        5             4         0
User4           0          0        4             5         0
User5           3          2        0             0         4

User Similarity Matrix:

          User1     User2    User3    User4     User5
User1  1.000000  0.978232  0.00000  0.00000  0.982873
User2  0.978232  1.000000  0.00000  0.00000  0.934646
User3  0.000000  0.000000  1.00000  0.97561  0.000000
User4  0.000000  0.000000  0.97561  1.00000  0.000000
User5  0.982873  0.934646  0.00000  0.00000  1.000000

Recommendations for User1:
[]

Recommendations for User3:
[]
